# Run Pipeline Evaluation & Analysis

Analysis of the outputs of `doc-grader` Tool against the established Human reference dataset, and plots the results.

Produces the `eval_gold_per_student.csv` and `eval_gold_summary.json` artefacts inside `outputs/gold_eval`, which are then loaded for visualisation in the subsequent cells.

In [ ]:
%load_ext autoreload
%autoreload 3

import logging
from pathlib import Path

from IPython.display import display
from scripts import eval
from scripts import eval_analysis as ea
from scripts import visual_utils as vu

from doc_grader.utils import configure_logging

configure_logging(logging.INFO)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("scripts.visual_utils").setLevel(logging.ERROR)
logging.getLogger("scripts.eval").setLevel(logging.INFO)

vu.configure_plot_style()

# Standard runs
par_dir = Path("../data/judge_gold/toolout/par")
int_dir = Path("../data/judge_gold/toolout_full/int")

# Model eval runs
eval_nm_dir = Path("../data/judge_gold/toolout_full/int_normal_judge_mini_content")
eval_mn_dir = Path("../data/judge_gold/toolout_full/int_mini_judge_nano_content")
eval_mnnl_dir = Path("../data/judge_gold/toolout_full/int_mini_judge_nano_content_nolocal")

gold_csv = Path("../data/judge_gold/gold_ipp_data.csv")
out_dir = Path("../outputs/gold_eval")

out_dir.mkdir(parents=True, exist_ok=True)

## 1. Run Evaluation
Runs the evaluation script directly if data is available, processing the LLM findings into comparable metrics.

In [ ]:
variant_dirs = {
    "par": par_dir if par_dir.exists() else None,
    "int": int_dir if int_dir.exists() else None,
    "int_normal_judge_mini_content": eval_nm_dir if eval_nm_dir.exists() else None,
    "int_mini_judge_nano_content": eval_mn_dir if eval_mn_dir.exists() else None,
    "int_mini_judge_nano_content_nolocal": eval_mnnl_dir if eval_mnnl_dir.exists() else None,
}

eval.evaluate_pipeline(
    variant_dirs=variant_dirs,
    gold_path=gold_csv if gold_csv.exists() else None,
    out_dir=out_dir,
    variant="all"
)

## 2. Load Evaluation Results

In [ ]:
per_student_df, summary = ea.load_eval_data(eval_dir=out_dir)
display(per_student_df.head())

## 3. Score Accuracy

### Score Scatter

Compares Tool Total Points vs Human Total Points for each document, normalised as a percentage of each task variant's max score.

In [ ]:
ax = ea.visualise_score_scatter(per_student_df)

### MAE by Score Quartile

Mean Absolute Error sliced by the actual document score. This shows if the tool struggles more on high-scoring or low-scoring documents.

In [ ]:
ax = ea.visualise_mae_by_score_quartile(per_student_df)

### Format Comparison

Compares PDF vs Markdown only on cost and latency.

By default this chart is filtered to the baseline full-eval aliases (`par`, `int`).
Use the scope table below to see exactly which model combinations are included in each alias.

In [ ]:
format_scope_df = ea.summarise_format_comparison_scope(per_student_df)

ax = ea.visualise_format_comparison(per_student_df)

## 4. Code Level Agreement

### Precision vs Recall Bubble Chart

How well the agent identifies specific deduction codes. Bubble size indicates how common the code is in the dataset.

In [ ]:
ax = ea.visualise_precision_recall_bubble(summary)

### Code Frequency Comparison

How often Tool applied a code vs how often Human did.

In [ ]:
ax = ea.visualise_code_frequency_comparison(summary)

### Per-Code Agreement

Agreement percentage by code.

In [ ]:
ax = ea.visualise_per_code_agreement(summary)

### Impact Delta Scatter

How far off the point deductions were for individual codes.

In [ ]:
ax = ea.visualise_impact_delta_scatter(per_student_df)

## 5. Pipeline Performance & Filtering

### Pipeline Waterfall

Shows how many putative findings enter the judge stage and how many survive to become final codes.

In [ ]:
ax = ea.visualise_pipeline_waterfall(summary)

### Judge Survival Rate

In [ ]:
ax = ea.visualise_judge_survival_rate(summary)

## 6. Cost and Latency

### Cost by Task Variant

In [ ]:
ax = ea.visualise_cost_by_variant(per_student_df)

### Cost by Generator Model

In [ ]:
ax = ea.visualise_cost_by_generator_model(per_student_df)

### Cost vs Document Score

In [ ]:
ax = ea.visualise_cost_vs_doc_points(per_student_df)

### Mean Token Usage by Task Variant (Table)

Token usage is shown as a table instead of a chart.

In [ ]:
token_usage_df = ea.summarise_token_usage_by_variant(per_student_df)
display(token_usage_df)

### Latency by Variant

### Latency by Generator Model

In [ ]:
ax = ea.visualise_latency_by_generator_model(per_student_df)

In [ ]:
ax = ea.visualise_latency_by_variant(per_student_df)

### Stage Times

In [ ]:
ax = ea.visualise_stage_times(per_student_df)

## 7. Aggregated Summaries

### Metrics per Variant Summary

In [ ]:
ax = ea.visualise_per_variant_metrics(summary)

### Metrics per Format Summary

Overall performance metrics split by document format (PDF vs Markdown).

In [ ]:
ax = ea.visualise_per_format_metrics(summary)

### Metrics per Language Summary

Overall performance metrics split by the student's document language.

In [ ]:
ax = ea.visualise_per_language_metrics(summary)

### Metrics per Generator Model

Overall performance metrics split by the content/asset AI model used.

In [ ]:
ax = ea.visualise_per_generator_model_metrics(summary)

### Metrics per Judge Model

Overall performance metrics split by the judge AI model used.

In [ ]:
ax = ea.visualise_per_judge_model_metrics(summary)